# 14 — RAG：讓 agent 開卷考試

**這份要學什麼**
- Embeddings（把文字變座標）+ VectorStore（依座標找最相關的資料）在做什麼
- 怎麼把「檢索」接成 graph 裡的一個普通 node，跟生成的 node 串起來

> 不需要 API key。用一個「誠實標註是假的」向量表示法做示範——機制（Embeddings / VectorStore /
> 檢索 / 注入 prompt）跟接真的 embedding 模型完全一樣，只是這裡的「相似度」是用文字重疊算的,
> 不是真的語意理解。

**這份要學什麼**
- Embeddings（把文字變成一串數字）跟 VectorStore（存這些數字、依相似度查詢）在做什麼
- `similarity_search` 怎麼運作
- 怎麼把「檢索」接成一個 node，塞進使用者問題前面再讓模型回答（RAG 的完整流程）
- 這跟前面章節的關係：檢索出來的資料就是一段文字，塞進 `SystemMessage` 就好，跟 `03`
  的 State/Node 概念沒有任何新東西，只是多了「怎麼找到相關資料」這一步

## 為什麼需要 RAG
LLM 回答問題靠的是訓練時記住的東西——像「閉卷考試」，考的都是背過的知識，考完之後公司內部
新增的規定、昨天才發生的事，它完全不知道。**RAG（Retrieval-Augmented Generation）**
把它變成「開卷考試」：考試前先讓它翻到課本裡最相關的那幾頁，再讓它照著回答——「翻書」
這個動作就是**檢索（retrieval）**，被翻到的那幾頁就是**上下文（context）**。

## Embeddings：把每段文字變成一個座標

**Embedding 模型**做的事，是把一段文字轉成一串數字（向量），你可以想成是幫每段文字在
一張地圖上標一個座標——意思相近的文字，座標就靠得近；意思差很遠的文字，座標就離很遠。
**VectorStore** 就是存這些座標的地方，`similarity_search(query)` 則是「把查詢也標一個
座標，找地圖上離它最近的幾個點」。

真的 embedding 模型（例如 OpenAI 的）需要 API key，這裡用 `notebooks/_rag.py` 裡的
`HashingBagOfCharsEmbeddings` 代替——它老實說了自己不是真的語意模型，只是依「文字重疊
程度」算座標（俗稱 hashing trick）。對這份 notebook 準備的範例資料來說，這樣已經足夠
讓相似度搜尋找到對的答案。

In [1]:
import sys

sys.path.insert(0, ".")
from _rag import HashingBagOfCharsEmbeddings

from langchain_core.vectorstores import InMemoryVectorStore

documents = [
    "LangGraph 的 checkpointer 用來儲存對話狀態，搭配 thread_id 可以恢復多輪對話。",
    "公司請假流程：先在系統送出申請，主管核准後才會生效，特休可以用半天為單位。",
    "InMemoryStore 是 LangGraph 的長期記憶機制，可以跨對話串存取使用者偏好。",
    "咖啡機的濾芯建議每三個月更換一次，避免影響出水口感。",
]

vector_store = InMemoryVectorStore(HashingBagOfCharsEmbeddings())
vector_store.add_texts(documents)

for query in ["怎麼用 checkpointer 恢復對話", "我想請特休", "咖啡機保養"]:
    top_doc = vector_store.similarity_search(query, k=1)[0]
    print(f"Q: {query}\n-> {top_doc.page_content}\n")

Q: 怎麼用 checkpointer 恢復對話
-> LangGraph 的 checkpointer 用來儲存對話狀態，搭配 thread_id 可以恢復多輪對話。

Q: 我想請特休
-> 公司請假流程：先在系統送出申請，主管核准後才會生效，特休可以用半天為單位。

Q: 咖啡機保養
-> 咖啡機的濾芯建議每三個月更換一次，避免影響出水口感。



三個完全不相關的問題，都翻到了對的那一頁——這就是 RAG 的核心能力：不用把所有資料塞進
prompt，只挑出跟問題相關的那一小段。

## 接進 graph：多一個 `retrieve` node

整個 RAG 流程用 `StateGraph` 表達，就是「檢索」跟「生成」兩個 node 接在一起：

1. **`retrieve`**：拿使用者的問題去 `similarity_search`，把找到的內容包成一則
   `SystemMessage` 塞進對話（讓模型知道「這是你可以參考的資料」）
2. **`generate`**：模型看著「參考資料 + 使用者問題」正常生成回答

沒有新概念——`retrieve` 就是 `03` 學過的一個普通 node，只是它做的事情是查 VectorStore
而不是呼叫 LLM。

In [2]:
from _llm import get_llm, has_api_key, scripted_model

from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph


def retrieve(state: MessagesState) -> dict:
    query = state["messages"][-1].content
    top_doc = vector_store.similarity_search(query, k=1)[0]
    return {"messages": [("system", f"參考資料：{top_doc.page_content}")]}


def generate(state: MessagesState) -> dict:
    model = scripted_model(["根據參考資料，checkpointer 是拿來存對話狀態、搭配 thread_id 恢復多輪對話用的。"])
    return {"messages": [model.invoke(state["messages"])]}


rag_builder = StateGraph(MessagesState)
rag_builder.add_node("retrieve", retrieve)
rag_builder.add_node("generate", generate)
rag_builder.add_edge(START, "retrieve")
rag_builder.add_edge("retrieve", "generate")
rag_builder.add_edge("generate", END)
rag_graph = rag_builder.compile()

rag_result = rag_graph.invoke({"messages": [HumanMessage("怎麼用 checkpointer 恢復對話？")]})
for m in rag_result["messages"]:
    print(f"{type(m).__name__:14} {m.content}")

HumanMessage   怎麼用 checkpointer 恢復對話？
SystemMessage  參考資料：LangGraph 的 checkpointer 用來儲存對話狀態，搭配 thread_id 可以恢復多輪對話。
AIMessage      根據參考資料，checkpointer 是拿來存對話狀態、搭配 thread_id 恢復多輪對話用的。


## 如果你有 API key：接真的 embedding 模型
換成 `OpenAIEmbeddings()`，其他程式碼（`InMemoryVectorStore`、`similarity_search`、
接進 graph 的方式）完全不用改——這正是 `Embeddings` 這層抽象的價值，跟 `13` 的
`get_llm()` 是同一個設計精神。

In [3]:
if has_api_key():
    from langchain_openai import OpenAIEmbeddings

    real_store = InMemoryVectorStore(OpenAIEmbeddings(model="text-embedding-3-small"))
    real_store.add_texts(documents)
    real_top = real_store.similarity_search("我想請特休", k=1)[0]
    print(real_top.page_content)
else:
    print("尚未設定 OPENAI_API_KEY，跳過真的 embedding 呼叫（上面已經展示了完整的檢索流程）。")

尚未設定 OPENAI_API_KEY，跳過真的 embedding 呼叫（上面已經展示了完整的檢索流程）。


## 正式環境：換一個真的向量資料庫
`InMemoryVectorStore` 跟這系列前面的 `InMemorySaver` / `InMemoryStore` 是同一個毛病：
只存在行程記憶體裡，程式一關就消失，也無法多台機器共用。正式環境常見的替代方案有
Chroma、pgvector（掛在 Postgres 上）、Pinecone、Redis（帶向量搜尋功能的版本）——
LangChain 幫每一種都包了對應的 `VectorStore` 實作，介面（`add_texts` / `similarity_search`
/ `as_retriever`）都跟這裡示範的一樣，換一行 import 加連線設定就能切換，不用重寫檢索邏輯。

## 小結
- RAG = 檢索（把問題變座標，找地圖上最近的資料）+ 生成（把找到的資料塞進 prompt 讓模型回答）
- 在 LangGraph 裡，檢索就是多一個 node，沒有新的圖概念——這也是為什麼 RAG 常常跟
  agent、多 agent（`09`）混在一起用：檢索本身可以是某個 specialist 的其中一個工具
- `Embeddings` / `VectorStore` 都是抽象介面，離線示範用的假 embedding 換成真的模型，
  其他程式碼不用動

下一份：`15_provider_sdks.ipynb`，剝開 LangChain 這層抽象，直接看原生 OpenAI /
Anthropic SDK 的工具呼叫格式。